# Notebook 07 — Final Evaluation & Model Selection
**Project:** Customer Churn Prediction  
**Depends on:** `02_preprocessing`, `03_baseline`, `05_tuning`, `06_xai_gate2`  
**Output:** `model_final.joblib` + `final_evaluation_results.json` + plots

Pipeline:
1. Load `xai_gate2_results.json` → ambil semua model `all_pass=True`
2. Evaluasi full metrics (PR-AUC, ROC-AUC, F1, Recall, Precision) pada val set
3. Threshold calibration (F1-optimal) per model
4. Perbandingan vs baseline model (pre-tuning)
5. Visualisasi: PR curve, ROC curve, Confusion Matrix grid
6. Ranking & seleksi best model
7. Export `model_final.joblib` + log ke WandB

---
## Install & Import

In [1]:
!pip install lightgbm xgboost wandb --quiet
print('OK install selesai.')

OK install selesai.


In [2]:
import os, json, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import joblib
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    f1_score, recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
import wandb

print('OK import selesai.')

OK import selesai.


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key    = user_secrets.get_secret('WANDB')
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ardiyanto24 (ardiyanto24-indonesian-national-police) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

---
## Konstanta Global

In [4]:
# ── Path ───────────────────────────────────────────────────────────────────────
PREPROCESSING_DIR = '/kaggle/input/notebooks/ardiyanto24/tccp-preprocessing-v2/artifacts'
BASELINE_DIR      = '/kaggle/input/notebooks/ardiyanto24/tccp-modeling-baseline-v2/artifacts'
TUNING_DIR        = '/kaggle/input/notebooks/ardiyanto24/tccp-hyperparameter-tuning/artifacts'
GATE2_DIR         = '/kaggle/input/notebooks/ardiyanto24/tccp-xai-gate-2/artifacts'

SPLITS_PATH       = f'{PREPROCESSING_DIR}/splits.joblib'
GATE2_JSON_PATH   = f'{GATE2_DIR}/xai_gate2_results.json'

OUTPUT_DIR = '/kaggle/working/artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── WandB ──────────────────────────────────────────────────────────────────────
WANDB_PROJECT     = 'customer-churn-prediction'
WANDB_EVAL_GROUP  = 'final-evaluation-07'

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Model selection ────────────────────────────────────────────────────────────
# Primary metric: PR-AUC (konsisten dengan tuning)
# Secondary metric: Recall_at_threshold (bisnis: false negative mahal)
PRIMARY_METRIC   = 'pr_auc'
SECONDARY_METRIC = 'recall'

# Kandidat (sama persis dengan Notebook 05/06, urutan berdasarkan tuned_pr_auc)
CANDIDATES = [
    {'model_name': 'lightgbm',            'balance': 'class_weight', 'tuned_pr_auc': 0.7522},
    {'model_name': 'xgboost',             'balance': 'class_weight', 'tuned_pr_auc': 0.7521},
    {'model_name': 'xgboost',             'balance': 'smote',        'tuned_pr_auc': 0.7485},
    {'model_name': 'lightgbm',            'balance': 'smote',        'tuned_pr_auc': 0.7477},
    {'model_name': 'logistic_regression', 'balance': 'class_weight', 'tuned_pr_auc': 0.7410},
    {'model_name': 'logistic_regression', 'balance': 'smote',        'tuned_pr_auc': 0.7406},
]

SEP = '=' * 65

print('OK konstanta global terdefinisi.')
print(f'   PRIMARY_METRIC   : {PRIMARY_METRIC}')
print(f'   SECONDARY_METRIC : {SECONDARY_METRIC}')
print(f'   Jumlah kandidat  : {len(CANDIDATES)} individual + 1 voting_ensemble')

OK konstanta global terdefinisi.
   PRIMARY_METRIC   : pr_auc
   SECONDARY_METRIC : recall
   Jumlah kandidat  : 6 individual + 1 voting_ensemble


---
## Class Definitions

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ArtifactLoader
# Tanggung jawab: load splits, gate2 results, tuned models, baseline
# ══════════════════════════════════════════════════════════════════════════════
class ArtifactLoader:
    """
    Load semua artifact yang dibutuhkan Notebook 07:
    - splits.joblib (val set)
    - xai_gate2_results.json (daftar model lulus)
    - tuned_<key>.joblib untuk setiap kandidat
    - baseline model (opsional — untuk perbandingan pre/post tuning)
    """

    def __init__(self):
        self.splits         = None
        self.gate2_results  = None
        self.tuned_models   = {}
        self.voting_ensemble = None
        self.baseline_model = None
        self.passed_keys    = []

    def load_splits(self):
        print('Loading splits...')
        self.splits        = joblib.load(SPLITS_PATH)
        self.X_val         = self.splits['X_val']
        self.y_val         = self.splits['y_val']
        self.feature_names = self.splits['feature_names']
        # Konversi ke DataFrame jika numpy
        if not isinstance(self.X_val, pd.DataFrame):
            self.X_val = pd.DataFrame(self.X_val, columns=self.feature_names)
        print(f'  X_val shape : {self.X_val.shape}')
        print(f'  y_val shape : {self.y_val.shape}')
        print(f'  Churn rate  : {self.y_val.mean():.4f}')

    def load_gate2_results(self):
        print('\nLoading gate2 results...')
        with open(GATE2_JSON_PATH) as f:
            self.gate2_results = json.load(f)
        self.passed_keys = self.gate2_results['passed_models']
        print(f'  Passed models : {len(self.passed_keys)}')
        for k in self.passed_keys:
            print(f'    - {k}')

    def load_tuned_models(self):
        print('\nLoading tuned models...')
        for cand in CANDIDATES:
            key   = f"{cand['model_name']}__{cand['balance']}"
            fpath = os.path.join(TUNING_DIR, f'tuned_{key}.joblib')
            if os.path.exists(fpath):
                self.tuned_models[key] = joblib.load(fpath)
                print(f'  OK {key}')
            else:
                print(f'  NOT FOUND: {fpath}')

        ens_path = os.path.join(TUNING_DIR, 'tuned_voting_ensemble.joblib')
        if os.path.exists(ens_path):
            self.voting_ensemble = joblib.load(ens_path)
            print('  OK voting_ensemble')
        else:
            print('  NOT FOUND: voting_ensemble')

    def load_baseline(self):
        """Load baseline model untuk perbandingan pre/post tuning (opsional)."""
        print('\nLoading baseline model (opsional)...')
        # Coba beberapa nama konvensional
        candidates_path = [
            os.path.join(BASELINE_DIR, 'best_baseline.joblib'),
            os.path.join(BASELINE_DIR, 'baseline_lightgbm__class_weight.joblib'),
            os.path.join(BASELINE_DIR, 'base_model.joblib'),
        ]
        for p in candidates_path:
            if os.path.exists(p):
                self.baseline_model = joblib.load(p)
                print(f'  OK baseline loaded: {os.path.basename(p)}')
                return
        print('  SKIP: baseline model tidak ditemukan (perbandingan akan dilewati)')

    def run(self):
        self.load_splits()
        self.load_gate2_results()
        self.load_tuned_models()
        self.load_baseline()
        return self

print('OK ArtifactLoader terdefinisi.')

OK ArtifactLoader terdefinisi.


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ThresholdCalibrator
# Tanggung jawab: temukan threshold optimal (F1-max) dari PR curve
# ══════════════════════════════════════════════════════════════════════════════
class ThresholdCalibrator:
    """
    Kalibrasi threshold prediksi menggunakan F1-optimal dari PR curve.

    Rationale bisnis:
    - False negative (missed churner) lebih mahal dari false positive
    - F1-optimal menyeimbangkan precision & recall
    - Recall dimonitor sebagai secondary check: jika recall < 0.60 pada
      threshold optimal, threshold diturunkan ke nilai yang menghasilkan
      recall=0.65 (recall floor)

    Output: threshold float yang digunakan untuk semua evaluasi berikutnya.
    """

    RECALL_FLOOR = 0.60   # minimum recall yang diterima

    def __init__(self, y_true, y_prob, model_key: str):
        self.y_true    = y_true
        self.y_prob    = y_prob
        self.model_key = model_key

    def run(self) -> dict:
        precisions, recalls, thresholds = precision_recall_curve(self.y_true, self.y_prob)

        # F1 untuk setiap threshold (thresholds array satu elemen lebih pendek)
        f1_scores = np.where(
            (precisions[:-1] + recalls[:-1]) > 0,
            2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
            0.0
        )

        best_idx       = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]
        best_f1        = f1_scores[best_idx]
        best_recall    = recalls[best_idx]
        best_precision = precisions[best_idx]
        calibration_mode = 'f1_optimal'

        # Cek recall floor — jika recall terlalu rendah, turunkan threshold
        if best_recall < self.RECALL_FLOOR:
            # Cari threshold di mana recall >= RECALL_FLOOR, ambil yang F1-nya tertinggi
            mask        = recalls[:-1] >= self.RECALL_FLOOR
            if mask.any():
                floor_idx      = np.argmax(f1_scores * mask)
                best_threshold = thresholds[floor_idx]
                best_f1        = f1_scores[floor_idx]
                best_recall    = recalls[floor_idx]
                best_precision = precisions[floor_idx]
                calibration_mode = f'recall_floor_{self.RECALL_FLOOR}'
                print(f'    [RECALL FLOOR] threshold disesuaikan ke recall≥{self.RECALL_FLOOR}')

        print(f'  Threshold      : {best_threshold:.4f}  [{calibration_mode}]')
        print(f'  F1             : {best_f1:.4f}')
        print(f'  Recall         : {best_recall:.4f}')
        print(f'  Precision      : {best_precision:.4f}')

        return {
            'threshold'       : float(best_threshold),
            'f1_at_threshold' : float(best_f1),
            'recall_at_threshold'   : float(best_recall),
            'precision_at_threshold': float(best_precision),
            'calibration_mode': calibration_mode,
            'pr_curve': {
                'precisions': precisions.tolist(),
                'recalls'   : recalls.tolist(),
                'thresholds': thresholds.tolist(),
            }
        }

print('OK ThresholdCalibrator terdefinisi.')

OK ThresholdCalibrator terdefinisi.


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ModelEvaluator
# Tanggung jawab: hitung full metrics untuk satu model pada val set
# ══════════════════════════════════════════════════════════════════════════════
class ModelEvaluator:
    """
    Evaluasi komprehensif satu kandidat model:
    - PR-AUC, ROC-AUC (threshold-independent)
    - Threshold calibration via ThresholdCalibrator
    - F1, Recall, Precision, Specificity, Accuracy di threshold optimal
    - Confusion matrix (TP, FP, TN, FN)
    - ROC curve data untuk plotting
    """

    def __init__(self, model_key: str, model,
                 X_val: pd.DataFrame, y_val,
                 tuned_pr_auc: float = None,
                 label: str = None):
        self.model_key    = model_key
        self.model        = model
        self.X_val        = X_val
        self.y_val        = np.array(y_val)
        self.tuned_pr_auc = tuned_pr_auc  # dari Notebook 05 (CV inner loop)
        self.label        = label or model_key

    def run(self) -> dict:
        print(f'\n{SEP}')
        print(f'  Evaluasi: {self.model_key}')
        print(SEP)

        t_start = time.time()

        # ── Prediksi probabilitas ─────────────────────────────────────────────
        y_prob = self.model.predict_proba(self.X_val)[:, 1]

        # ── Threshold-independent metrics ─────────────────────────────────────
        pr_auc  = average_precision_score(self.y_val, y_prob)
        roc_auc = roc_auc_score(self.y_val, y_prob)
        print(f'  PR-AUC  (val)  : {pr_auc:.4f}', end='')
        if self.tuned_pr_auc:
            delta = pr_auc - self.tuned_pr_auc
            print(f'  (tuning CV: {self.tuned_pr_auc:.4f}, delta: {delta:+.4f})')
        else:
            print()
        print(f'  ROC-AUC (val)  : {roc_auc:.4f}')

        # ── Threshold calibration ─────────────────────────────────────────────
        print('\n  [Threshold Calibration]')
        cal = ThresholdCalibrator(
            y_true    = self.y_val,
            y_prob    = y_prob,
            model_key = self.model_key,
        ).run()
        threshold = cal['threshold']

        # ── Threshold-dependent metrics ───────────────────────────────────────
        y_pred = (y_prob >= threshold).astype(int)
        f1        = f1_score(self.y_val, y_pred)
        recall    = recall_score(self.y_val, y_pred)
        precision = precision_score(self.y_val, y_pred)
        tn, fp, fn, tp = confusion_matrix(self.y_val, y_pred).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        accuracy    = (tp + tn) / len(self.y_val)

        # ROC curve data
        fpr_arr, tpr_arr, _ = roc_curve(self.y_val, y_prob)

        elapsed = time.time() - t_start
        print(f'\n  [Metrics @ threshold={threshold:.4f}]')
        print(f'  F1          : {f1:.4f}')
        print(f'  Recall      : {recall:.4f}')
        print(f'  Precision   : {precision:.4f}')
        print(f'  Specificity : {specificity:.4f}')
        print(f'  Accuracy    : {accuracy:.4f}')
        print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
        print(f'  Waktu       : {elapsed:.1f}s')

        return {
            'key'           : self.model_key,
            'label'         : self.label,
            'pr_auc'        : float(pr_auc),
            'roc_auc'       : float(roc_auc),
            'tuned_pr_auc'  : self.tuned_pr_auc,
            'threshold'     : float(threshold),
            'calibration_mode': cal['calibration_mode'],
            'f1'            : float(f1),
            'recall'        : float(recall),
            'precision'     : float(precision),
            'specificity'   : float(specificity),
            'accuracy'      : float(accuracy),
            'tp'            : int(tp),
            'fp'            : int(fp),
            'tn'            : int(tn),
            'fn'            : int(fn),
            # Curve data untuk plotting
            '_y_prob'       : y_prob,
            '_y_pred'       : y_pred,
            '_pr_curve'     : cal['pr_curve'],
            '_roc_fpr'      : fpr_arr,
            '_roc_tpr'      : tpr_arr,
        }

print('OK ModelEvaluator terdefinisi.')

OK ModelEvaluator terdefinisi.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: CurvePlotter
# Tanggung jawab: plot PR curve dan ROC curve semua model dalam satu figure
# ══════════════════════════════════════════════════════════════════════════════
class CurvePlotter:
    """
    Plot perbandingan PR curve dan ROC curve untuk semua kandidat.
    Model terbaik (best_key) diberi highlight warna berbeda.
    """

    # Palet warna: 7 model, voting ensemble selalu merah
    PALETTE = [
        '#534AB7', '#2E86AB', '#A23B72', '#F18F01',
        '#C73E1D', '#3B1F2B', '#44BBA4',
    ]

    def __init__(self, eval_results: list, best_key: str,
                 y_val, baseline_result: dict = None):
        self.eval_results    = eval_results
        self.best_key        = best_key
        self.y_val           = np.array(y_val)
        self.baseline_result = baseline_result

    def _get_color(self, idx: int, key: str) -> str:
        if key == self.best_key:
            return '#E63946'   # merah — best model
        return self.PALETTE[idx % len(self.PALETTE)]

    def _get_lw(self, key: str) -> float:
        return 2.5 if key == self.best_key else 1.2

    def plot_pr_curves(self) -> str:
        fig, ax = plt.subplots(figsize=(9, 6))

        baseline_pr = (self.y_val == 1).mean()
        ax.axhline(y=baseline_pr, color='gray', linestyle='--',
                   linewidth=1, label=f'Random baseline (PR={baseline_pr:.3f})')

        for idx, r in enumerate(self.eval_results):
            prc    = r['_pr_curve']
            color  = self._get_color(idx, r['key'])
            lw     = self._get_lw(r['key'])
            marker = '★' if r['key'] == self.best_key else ''
            label  = f"{marker}{r['key']} (PR-AUC={r['pr_auc']:.4f})"
            ax.plot(prc['recalls'], prc['precisions'],
                    color=color, linewidth=lw, label=label,
                    alpha=0.9 if r['key'] == self.best_key else 0.7)

        if self.baseline_result:
            r = self.baseline_result
            prc = r['_pr_curve']
            ax.plot(prc['recalls'], prc['precisions'],
                    color='#B4B2A9', linewidth=1.2, linestyle=':',
                    label=f"baseline_pre_tuning (PR-AUC={r['pr_auc']:.4f})")

        ax.set_xlabel('Recall', fontsize=12)
        ax.set_ylabel('Precision', fontsize=12)
        ax.set_title('Precision–Recall Curve — Semua Kandidat (Val Set)', fontsize=13, fontweight='bold')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

        path = os.path.join(OUTPUT_DIR, 'eval_pr_curves.png')
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'PR curve disimpan: {path}')
        return path

    def plot_roc_curves(self) -> str:
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot([0, 1], [0, 1], color='gray', linestyle='--',
                linewidth=1, label='Random baseline (AUC=0.500)')

        for idx, r in enumerate(self.eval_results):
            color  = self._get_color(idx, r['key'])
            lw     = self._get_lw(r['key'])
            marker = '★' if r['key'] == self.best_key else ''
            label  = f"{marker}{r['key']} (ROC-AUC={r['roc_auc']:.4f})"
            ax.plot(r['_roc_fpr'], r['_roc_tpr'],
                    color=color, linewidth=lw, label=label,
                    alpha=0.9 if r['key'] == self.best_key else 0.7)

        if self.baseline_result:
            r = self.baseline_result
            ax.plot(r['_roc_fpr'], r['_roc_tpr'],
                    color='#B4B2A9', linewidth=1.2, linestyle=':',
                    label=f"baseline_pre_tuning (ROC-AUC={r['roc_auc']:.4f})")

        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title('ROC Curve — Semua Kandidat (Val Set)', fontsize=13, fontweight='bold')
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

        path = os.path.join(OUTPUT_DIR, 'eval_roc_curves.png')
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'ROC curve disimpan: {path}')
        return path

print('OK CurvePlotter terdefinisi.')

OK CurvePlotter terdefinisi.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ConfusionMatrixPlotter
# Tanggung jawab: grid confusion matrix semua kandidat + baseline
# ══════════════════════════════════════════════════════════════════════════════
class ConfusionMatrixPlotter:
    """
    Plot grid confusion matrix untuk semua model yang dievaluasi.
    Setiap subplot = satu model, nilai absolut + persen.
    """

    def __init__(self, eval_results: list, y_val, best_key: str,
                 baseline_result: dict = None):
        self.eval_results    = eval_results
        self.y_val           = np.array(y_val)
        self.best_key        = best_key
        self.baseline_result = baseline_result

    def run(self) -> str:
        all_results = list(self.eval_results)
        if self.baseline_result:
            all_results = [self.baseline_result] + all_results

        n       = len(all_results)
        ncols   = 4
        nrows   = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
        axes_flat = axes.flatten() if n > 1 else [axes]

        for i, r in enumerate(all_results):
            ax = axes_flat[i]
            cm = np.array([[r['tn'], r['fp']], [r['fn'], r['tp']]])
            total = cm.sum()
            disp  = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=['No Churn', 'Churn']
            )
            disp.plot(ax=ax, colorbar=False, cmap='Blues')

            # Tambahkan persentase
            for text_obj in ax.texts:
                val = int(float(text_obj.get_text()))
                text_obj.set_text(f'{val}\n({val/total:.1%})')
                text_obj.set_fontsize(8)

            title = r['key']
            if r['key'] == self.best_key:
                title = f'★ {title}'
                ax.title.set_color('#E63946')
            ax.set_title(title, fontsize=8, fontweight='bold')
            ax.set_xlabel('Predicted', fontsize=8)
            ax.set_ylabel('Actual', fontsize=8)

            # Annotasi metric ringkas
            ax.text(0.5, -0.22,
                    f"Recall={r['recall']:.3f}  F1={r['f1']:.3f}  Prec={r['precision']:.3f}",
                    ha='center', fontsize=7, transform=ax.transAxes, color='#333333')

        # Sembunyikan subplot kosong
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)

        fig.suptitle('Confusion Matrix — Semua Kandidat @ Threshold Optimal',
                     fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()

        path = os.path.join(OUTPUT_DIR, 'eval_confusion_matrices.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Confusion matrix grid disimpan: {path}')
        return path

print('OK ConfusionMatrixPlotter terdefinisi.')

OK ConfusionMatrixPlotter terdefinisi.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ModelRanker
# Tanggung jawab: ranking semua model, cetak tabel, tentukan best model
# ══════════════════════════════════════════════════════════════════════════════
class ModelRanker:
    """
    Ranking model berdasarkan:
    1. PRIMARY_METRIC  : pr_auc  (threshold-independent, konsisten dengan tuning)
    2. SECONDARY_METRIC: recall  (false negative costly untuk bisnis churn)

    Voting Ensemble diikutsertakan dalam ranking — jika ia model terbaik,
    ia yang masuk deployment. Jika tidak, individual model yang dipilih.

    Output: key string dari best model.
    """

    COLS_DISPLAY = [
        'key', 'pr_auc', 'roc_auc', 'f1', 'recall', 'precision',
        'specificity', 'threshold', 'tp', 'fp', 'tn', 'fn',
    ]

    def __init__(self, eval_results: list, baseline_result: dict = None):
        self.eval_results    = eval_results
        self.baseline_result = baseline_result

    def _make_df(self, results) -> pd.DataFrame:
        rows = []
        for r in results:
            rows.append({c: r[c] for c in self.COLS_DISPLAY if c in r})
        return pd.DataFrame(rows)

    def run(self) -> str:
        print(f'\n{SEP}')
        print('  RANKING MODEL — FINAL EVALUATION')
        print(SEP)

        df = self._make_df(self.eval_results)
        df_sorted = df.sort_values(
            [PRIMARY_METRIC, SECONDARY_METRIC],
            ascending=False
        ).reset_index(drop=True)

        df_sorted.insert(0, 'Rank', range(1, len(df_sorted) + 1))

        # Format untuk display
        float_cols = ['pr_auc', 'roc_auc', 'f1', 'recall', 'precision',
                      'specificity', 'threshold']
        df_display = df_sorted.copy()
        for col in float_cols:
            df_display[col] = df_display[col].map(lambda x: f'{x:.4f}')

        print(df_display[['Rank', 'key', 'pr_auc', 'roc_auc', 'f1',
                           'recall', 'precision', 'threshold']].to_string(index=False))

        best_key = df_sorted.iloc[0]['key']
        best_row = df_sorted.iloc[0]

        print(f'\n{SEP}')
        print(f'  ★ BEST MODEL: {best_key}')
        print(f'  PR-AUC      : {best_row["pr_auc"]:.4f}')
        print(f'  ROC-AUC     : {best_row["roc_auc"]:.4f}')
        print(f'  F1          : {best_row["f1"]:.4f}')
        print(f'  Recall      : {best_row["recall"]:.4f}')
        print(f'  Precision   : {best_row["precision"]:.4f}')
        print(f'  Threshold   : {best_row["threshold"]:.4f}')
        print(SEP)

        # Baseline comparison (jika ada)
        if self.baseline_result:
            b = self.baseline_result
            b_best_pr = best_row['pr_auc']
            delta_pr  = b_best_pr - b['pr_auc']
            print(f'\n  Baseline (pre-tuning) PR-AUC : {b["pr_auc"]:.4f}')
            print(f'  Best model    PR-AUC         : {b_best_pr:.4f}')
            print(f'  Delta (tuning gain)          : {delta_pr:+.4f}')

        return best_key, df_sorted

print('OK ModelRanker terdefinisi.')

OK ModelRanker terdefinisi.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ModelExporter
# Tanggung jawab: simpan model_final.joblib + log ke WandB
# ══════════════════════════════════════════════════════════════════════════════
class ModelExporter:
    """
    Export best model:
    1. Simpan model ke model_final.joblib
    2. Simpan metadata (key, threshold, metrics) ke model_final_meta.json
    3. Log semua metrics + artifacts ke WandB
    4. Log semua plot yang sudah dibuat ke WandB
    """

    def __init__(self, best_key: str, best_model,
                 best_result: dict, all_eval_results: list,
                 plot_paths: list):
        self.best_key         = best_key
        self.best_model       = best_model
        self.best_result      = best_result
        self.all_eval_results = all_eval_results
        self.plot_paths       = plot_paths

    def _make_serializable(self, obj):
        import numpy as np
        if isinstance(obj, (np.bool_,)):    return bool(obj)
        if isinstance(obj, bool):           return bool(obj)
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        if isinstance(obj, dict):
            return {k: self._make_serializable(v) for k, v in obj.items()
                    if not k.startswith('_')}  # skip internal keys (y_prob, dll)
        if isinstance(obj, list):           return [self._make_serializable(i) for i in obj]
        return obj

    def save_artifacts(self):
        # ── 1. model_final.joblib ─────────────────────────────────────────────
        model_path = os.path.join(OUTPUT_DIR, 'model_final.joblib')
        joblib.dump(self.best_model, model_path)
        print(f'OK model_final.joblib disimpan: {model_path}')

        # ── 2. model_final_meta.json ──────────────────────────────────────────
        meta = {
            'notebook'    : 'Notebook 07 — Final Evaluation',
            'best_model'  : self.best_key,
            'threshold'   : self.best_result['threshold'],
            'pr_auc'      : self.best_result['pr_auc'],
            'roc_auc'     : self.best_result['roc_auc'],
            'f1'          : self.best_result['f1'],
            'recall'      : self.best_result['recall'],
            'precision'   : self.best_result['precision'],
            'specificity' : self.best_result['specificity'],
            'accuracy'    : self.best_result['accuracy'],
            'tp'          : self.best_result['tp'],
            'fp'          : self.best_result['fp'],
            'tn'          : self.best_result['tn'],
            'fn'          : self.best_result['fn'],
            'calibration_mode': self.best_result['calibration_mode'],
        }
        meta_path = os.path.join(OUTPUT_DIR, 'model_final_meta.json')
        with open(meta_path, 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'OK model_final_meta.json disimpan: {meta_path}')

        return model_path, meta_path

    def log_to_wandb(self, model_path: str, meta_path: str):
        run = wandb.init(
            project = WANDB_PROJECT,
            group   = WANDB_EVAL_GROUP,
            name    = f'final_eval__{self.best_key}',
            config  = {
                'best_model'      : self.best_key,
                'primary_metric'  : PRIMARY_METRIC,
                'secondary_metric': SECONDARY_METRIC,
            },
        )

        # Log metrics best model
        wandb.log({
            'best/pr_auc'     : self.best_result['pr_auc'],
            'best/roc_auc'    : self.best_result['roc_auc'],
            'best/f1'         : self.best_result['f1'],
            'best/recall'     : self.best_result['recall'],
            'best/precision'  : self.best_result['precision'],
            'best/specificity': self.best_result['specificity'],
            'best/accuracy'   : self.best_result['accuracy'],
            'best/threshold'  : self.best_result['threshold'],
        })

        # Log metrics semua model
        for r in self.all_eval_results:
            prefix = r['key'].replace('__', '_')
            wandb.log({
                f'{prefix}/pr_auc'   : r['pr_auc'],
                f'{prefix}/roc_auc'  : r['roc_auc'],
                f'{prefix}/f1'       : r['f1'],
                f'{prefix}/recall'   : r['recall'],
                f'{prefix}/precision': r['precision'],
                f'{prefix}/threshold': r['threshold'],
            })

        # Log plots
        for p in self.plot_paths:
            if p and os.path.exists(p):
                wandb.log({os.path.basename(p): wandb.Image(p)})

        # Log model artifact
        artifact = wandb.Artifact('model_final', type='model')
        artifact.add_file(model_path)
        artifact.add_file(meta_path)
        wandb.log_artifact(artifact)

        wandb.finish()
        print('OK WandB logging selesai.')

    def run(self):
        model_path, meta_path = self.save_artifacts()
        self.log_to_wandb(model_path, meta_path)
        return model_path, meta_path

print('OK ModelExporter terdefinisi.')
print()
print('OK Semua class terdefinisi. Siap eksekusi.')

OK ModelExporter terdefinisi.

OK Semua class terdefinisi. Siap eksekusi.


---
## Load Artifacts

In [12]:
loader = ArtifactLoader().run()

X_val         = loader.X_val
y_val         = loader.y_val
feature_names = loader.feature_names
tuned_models  = loader.tuned_models
voting_ens    = loader.voting_ensemble
baseline_mdl  = loader.baseline_model
passed_keys   = loader.passed_keys

# Map candidate ke tuned_pr_auc dari Notebook 05
pr_auc_map = {f"{c['model_name']}__{c['balance']}": c['tuned_pr_auc'] for c in CANDIDATES}
pr_auc_map['voting_ensemble'] = None  # tidak ada single tuned_pr_auc untuk ensemble

print(f'\nTotal model siap evaluasi : {len(tuned_models) + (1 if voting_ens else 0)}')

Loading splits...
  X_val shape : (89129, 29)
  y_val shape : (89129,)
  Churn rate  : 0.2252

Loading gate2 results...
  Passed models : 7
    - lightgbm__class_weight
    - xgboost__class_weight
    - xgboost__smote
    - lightgbm__smote
    - logistic_regression__class_weight
    - logistic_regression__smote
    - voting_ensemble

Loading tuned models...
  OK lightgbm__class_weight
  OK xgboost__class_weight
  OK xgboost__smote
  OK lightgbm__smote
  OK logistic_regression__class_weight
  OK logistic_regression__smote
  OK voting_ensemble

Loading baseline model (opsional)...
  SKIP: baseline model tidak ditemukan (perbandingan akan dilewati)

Total model siap evaluasi : 7


---
## Evaluasi Baseline (Pre-Tuning)

In [13]:
baseline_result = None

if baseline_mdl is not None:
    print('Evaluasi baseline model (pre-tuning)...')
    baseline_result = ModelEvaluator(
        model_key    = 'baseline_pre_tuning',
        model        = baseline_mdl,
        X_val        = X_val,
        y_val        = y_val,
        tuned_pr_auc = None,
        label        = 'baseline_pre_tuning',
    ).run()
    print('\nOK baseline evaluation selesai.')
else:
    print('SKIP: baseline model tidak tersedia. Perbandingan pre/post tuning dilewati.')

SKIP: baseline model tidak tersedia. Perbandingan pre/post tuning dilewati.


---
## Evaluasi Semua Kandidat (Post-Tuning)

In [14]:
all_eval_results = []

# ── Individual models ─────────────────────────────────────────────────────────
for cand in CANDIDATES:
    key = f"{cand['model_name']}__{cand['balance']}"
    if key not in tuned_models:
        print(f'SKIP {key}: model tidak ditemukan')
        continue
    if key not in passed_keys:
        print(f'SKIP {key}: tidak lulus XAI Gate #2')
        continue

    result = ModelEvaluator(
        model_key    = key,
        model        = tuned_models[key],
        X_val        = X_val,
        y_val        = y_val,
        tuned_pr_auc = pr_auc_map.get(key),
    ).run()
    all_eval_results.append(result)

# ── Voting Ensemble ───────────────────────────────────────────────────────────
if voting_ens is not None and 'voting_ensemble' in passed_keys:
    result = ModelEvaluator(
        model_key    = 'voting_ensemble',
        model        = voting_ens,
        X_val        = X_val,
        y_val        = y_val,
        tuned_pr_auc = None,
    ).run()
    all_eval_results.append(result)

print(f'\nOK total model dievaluasi: {len(all_eval_results)}')


  Evaluasi: lightgbm__class_weight
  PR-AUC  (val)  : 0.7522  (tuning CV: 0.7522, delta: -0.0000)
  ROC-AUC (val)  : 0.9153

  [Threshold Calibration]
  Threshold      : 0.6565  [f1_optimal]
  F1             : 0.7018
  Recall         : 0.7828
  Precision      : 0.6359

  [Metrics @ threshold=0.6565]
  F1          : 0.7018
  Recall      : 0.7828
  Precision   : 0.6359
  Specificity : 0.8697
  Accuracy    : 0.8501
  TP=15713  FP=8997  TN=60060  FN=4359
  Waktu       : 1.8s

  Evaluasi: xgboost__class_weight
  PR-AUC  (val)  : 0.7521  (tuning CV: 0.7521, delta: +0.0000)
  ROC-AUC (val)  : 0.9152

  [Threshold Calibration]
  Threshold      : 0.6664  [f1_optimal]
  F1             : 0.7017
  Recall         : 0.7708
  Precision      : 0.6439

  [Metrics @ threshold=0.6664]
  F1          : 0.7017
  Recall      : 0.7708
  Precision   : 0.6439
  Specificity : 0.8761
  Accuracy    : 0.8524
  TP=15472  FP=8557  TN=60500  FN=4600
  Waktu       : 1.1s

  Evaluasi: xgboost__smote
  PR-AUC  (val)  : 

---
## Ranking & Best Model Selection

In [15]:
ranker = ModelRanker(
    eval_results    = all_eval_results,
    baseline_result = baseline_result,
)
best_key, ranking_df = ranker.run()

# Ambil best_model object dan best_result dict
best_result = next(r for r in all_eval_results if r['key'] == best_key)
if best_key == 'voting_ensemble':
    best_model = voting_ens
else:
    best_model = tuned_models[best_key]


  RANKING MODEL — FINAL EVALUATION
 Rank                               key pr_auc roc_auc     f1 recall precision threshold
    1                   voting_ensemble 0.7525  0.9155 0.7025 0.7820    0.6377    0.6238
    2            lightgbm__class_weight 0.7522  0.9153 0.7018 0.7828    0.6359    0.6565
    3             xgboost__class_weight 0.7521  0.9152 0.7017 0.7708    0.6439    0.6664
    4                    xgboost__smote 0.7485  0.9134 0.6978 0.7586    0.6460    0.6385
    5                   lightgbm__smote 0.7477  0.9134 0.6979 0.7685    0.6392    0.5986
    6 logistic_regression__class_weight 0.7410  0.9122 0.6954 0.7818    0.6262    0.6662
    7        logistic_regression__smote 0.7406  0.9121 0.6954 0.7732    0.6318    0.6782

  ★ BEST MODEL: voting_ensemble
  PR-AUC      : 0.7525
  ROC-AUC     : 0.9155
  F1          : 0.7025
  Recall      : 0.7820
  Precision   : 0.6377
  Threshold   : 0.6238


---
## Visualisasi PR Curve

In [16]:
plotter = CurvePlotter(
    eval_results    = all_eval_results,
    best_key        = best_key,
    y_val           = y_val,
    baseline_result = baseline_result,
)
pr_curve_path = plotter.plot_pr_curves()

PR curve disimpan: /kaggle/working/artifacts/eval_pr_curves.png


---
## Visualisasi ROC Curve

In [17]:
roc_curve_path = plotter.plot_roc_curves()

ROC curve disimpan: /kaggle/working/artifacts/eval_roc_curves.png


---
## Visualisasi Confusion Matrix Grid

In [18]:
cm_path = ConfusionMatrixPlotter(
    eval_results    = all_eval_results,
    y_val           = y_val,
    best_key        = best_key,
    baseline_result = baseline_result,
).run()

Confusion matrix grid disimpan: /kaggle/working/artifacts/eval_confusion_matrices.png


---
## Export Best Model & WandB Logging

In [19]:
exporter = ModelExporter(
    best_key         = best_key,
    best_model       = best_model,
    best_result      = best_result,
    all_eval_results = all_eval_results,
    plot_paths       = [pr_curve_path, roc_curve_path, cm_path],
)
model_path, meta_path = exporter.run()

OK model_final.joblib disimpan: /kaggle/working/artifacts/model_final.joblib
OK model_final_meta.json disimpan: /kaggle/working/artifacts/model_final_meta.json


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260408_053125-n0r44f4m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run final_eval__voting_ensemble
wandb: ⭐️ View project at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction
wandb: 🚀 View run at https://wandb.ai/ardiyanto24-indonesian-national-police/customer-churn-prediction/runs/n0r44f4m
wandb: updating run metadata; uploading artifact model_final
wandb: uploading artifact model_final
wandb: uploading media/images/eval_roc_curves.png_9_79384303d515e4a3e41a.png; uploading media/images/eval_confusion_matrices.png_10_802a6b2971382730ac9f.png; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json (+ 2 more)
wandb: 
wandb: Run history:
wandb:                best/accuracy ▁
wandb:                      best/f1 ▁
wandb:                  best/pr_auc ▁
wandb:               best/precision ▁
wandb:          

OK WandB logging selesai.


---
## Simpan Output untuk Notebook 08 (Deployment)

In [20]:
def make_serializable(obj):
    """Konversi numpy types ke Python native untuk JSON."""
    if isinstance(obj, (np.bool_,)):    return bool(obj)
    if isinstance(obj, bool):           return bool(obj)
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()
                if not k.startswith('_')}
    if isinstance(obj, list):           return [make_serializable(i) for i in obj]
    return obj

output = {
    'notebook'        : 'Notebook 07 — Final Evaluation',
    'best_model'      : best_key,
    'best_model_path' : model_path,
    'threshold'       : best_result['threshold'],
    'calibration_mode': best_result['calibration_mode'],
    'final_metrics'   : {
        'pr_auc'      : best_result['pr_auc'],
        'roc_auc'     : best_result['roc_auc'],
        'f1'          : best_result['f1'],
        'recall'      : best_result['recall'],
        'precision'   : best_result['precision'],
        'specificity' : best_result['specificity'],
        'accuracy'    : best_result['accuracy'],
    },
    'all_results'     : [make_serializable(r) for r in all_eval_results],
    'baseline_result' : make_serializable(baseline_result) if baseline_result else None,
    'ranking'         : ranking_df[['key', 'pr_auc', 'roc_auc', 'f1', 'recall',
                                    'precision', 'threshold']].to_dict(orient='records'),
}

out_path = os.path.join(OUTPUT_DIR, 'final_evaluation_results.json')
with open(out_path, 'w') as f:
    json.dump(make_serializable(output), f, indent=2)

print(f'OK final_evaluation_results.json disimpan: {out_path}')
print(json.dumps({
    'best_model': best_key,
    'threshold' : round(best_result['threshold'], 4),
    'pr_auc'    : round(best_result['pr_auc'], 4),
    'recall'    : round(best_result['recall'], 4),
    'f1'        : round(best_result['f1'], 4),
}, indent=2))

OK final_evaluation_results.json disimpan: /kaggle/working/artifacts/final_evaluation_results.json
{
  "best_model": "voting_ensemble",
  "threshold": 0.6238,
  "pr_auc": 0.7525,
  "recall": 0.782,
  "f1": 0.7025
}


---
## Ringkasan Final

In [21]:
import glob

print(SEP)
print('  NOTEBOOK 07 SELESAI')
print(SEP)
print()
print(f'  Total kandidat dievaluasi : {len(all_eval_results)}')
print(f'  Best model terpilih       : {best_key}')
print()
print('  Final metrics (val set):')
print(f'    PR-AUC      : {best_result["pr_auc"]:.4f}')
print(f'    ROC-AUC     : {best_result["roc_auc"]:.4f}')
print(f'    F1          : {best_result["f1"]:.4f}')
print(f'    Recall      : {best_result["recall"]:.4f}')
print(f'    Precision   : {best_result["precision"]:.4f}')
print(f'    Threshold   : {best_result["threshold"]:.4f}  [{best_result["calibration_mode"]}]')

if baseline_result:
    gain = best_result['pr_auc'] - baseline_result['pr_auc']
    print()
    print(f'  Tuning gain (PR-AUC)      : {gain:+.4f}')
    print(f'  Baseline PR-AUC           : {baseline_result["pr_auc"]:.4f}')

print()
print('  Artifacts:')
for fpath in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {os.path.basename(fpath):<55} ({size_kb:.0f} KB)')
print()
print('  Langkah berikutnya: Deployment (FastAPI + Streamlit)')
print('    Load model_final.joblib + preprocessor.joblib')
print('    Gunakan threshold dari model_final_meta.json')
print('    Inference: preprocess → predict_proba → apply threshold')
print(SEP)

  NOTEBOOK 07 SELESAI

  Total kandidat dievaluasi : 7
  Best model terpilih       : voting_ensemble

  Final metrics (val set):
    PR-AUC      : 0.7525
    ROC-AUC     : 0.9155
    F1          : 0.7025
    Recall      : 0.7820
    Precision   : 0.6377
    Threshold   : 0.6238  [f1_optimal]

  Artifacts:
  eval_confusion_matrices.png                             (133 KB)
  eval_pr_curves.png                                      (132 KB)
  eval_roc_curves.png                                     (134 KB)
  final_evaluation_results.json                           (6 KB)
  model_final.joblib                                      (24901 KB)
  model_final_meta.json                                   (0 KB)

  Langkah berikutnya: Deployment (FastAPI + Streamlit)
    Load model_final.joblib + preprocessor.joblib
    Gunakan threshold dari model_final_meta.json
    Inference: preprocess → predict_proba → apply threshold
